# From the policy gradient to PPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/reinforcement_learning/policy_gradient_to_ppo.ipynb)

Policy methods adjust the behaviour directly: a stochastic policy $\pi_\theta(u \mid x)$ with parameters $\theta$, moved in the direction that lowers the expected cost. This notebook walks the chain from that idea to PPO, the algorithm behind the swing-up and drone notebooks: the parameterized policy, the policy gradient theorem, REINFORCE, the baseline and the advantage, actor-critic, generalized advantage estimation, and the clipped objective.

Every formula is checked numerically against the function that implements it in minilink's reinforcement-learning planner, so the code is the mathematics on this page. The last section lets the planner assemble those same pieces and learn a pendulum swing-up.

**Two conventions.** The course minimizes a cost: the stage cost of one control period is $c_k = g(x_k, u_k)\,\Delta t$, the discount is $\alpha$, the cost-to-go of a policy is $J^\pi(x)$ and the cost of an action is $Q^\pi(x, u)$. RL code maximizes a reward, $r_k = -c_k$, with a discount $\gamma = \alpha$ and a value $V^\pi = -J^\pi$. The planner uses the reward form internally; each section states the formula in cost form and notes where the sign flips.

The planner API is in [`11_reinforcement_learning`](../../tutorial/11_reinforcement_learning.ipynb); the environment interface in [`gymnasium_interface`](gymnasium_interface.ipynb).

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## 1. The task as a Markov decision process

The pendulum swing-up: $x_{k+1} = f(x_k, u_k)$ over one control period with the torque held, a stage cost $c_k = g(x_k, u_k)\,\Delta t$ that is zero upright and grows toward hanging, and a discount $\alpha$ that weighs the future over a horizon of about $\Delta t / (-\ln \alpha)$ seconds. The objective of a policy is

$$J(\theta) = \mathbb{E}_{\pi_\theta}\Big[\sum_{k=0}^{\infty} \alpha^k c_k\Big].$$

The planner below is only built here, not trained: its rollout environment, its Gaussian exploration head and its PPO update rule are the objects the next sections examine. One step of the environment returns the reward $r = -c$.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from minilink import CostFunction, Pendulum
from minilink.control import angle_features
from minilink.planning import (
    MonteCarloEvaluator,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)
from minilink.planning.reinforcement_learning import gae

TORQUE = 4.0  # Nm, below m g l = 9.81 Nm
DT = 0.05  # control period

plant = Pendulum()
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([TORQUE])
plant.state.lower_bound = np.array([-4 * np.pi, -20.0])
plant.state.upper_bound = np.array([4 * np.pi, 20.0])


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + jnp.cos(theta)) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


cost = SwingUpCost()
problem = StochasticPlanningProblem(
    plant, cost=cost, tf=np.inf, x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0])
)
planner = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=angle_features(angles=[0], scales={1: 0.1}),
    hidden=(32, 32),
    n_envs=64,
    n_steps=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
    verbose=0,
)
env, head, ppo = planner.env, planner.head, planner.algorithm

# One control period, and its reward
x, u = jnp.array([0.05, 0.0]), jnp.array([TORQUE])
x_next, t_next, r, terminated, truncated = env.step(x, 0.0, u, jax.random.PRNGKey(0))
c = float(cost.g(x, u)) * DT
print(f"stage cost c = g(x, u) dt = {c:.5f}   reward r = {float(r):.5f}")
print(f"discount alpha = {planner.gamma}: a horizon of {DT / -np.log(planner.gamma):.1f} s")

## 2. A parameterized stochastic policy

A stochastic policy is a probability distribution over actions, $\pi_\theta(u \mid x) = P(u_k = u \mid x_k = x, \theta)$. For continuous torques the usual choice is a Gaussian around a mean given by a neural network,

$$u \sim \mathcal{N}\big(\mu_\theta(x), \sigma^2\big), \qquad \ln \pi_\theta(u \mid x) = \sum_i \Big[ -\frac{(u_i - \mu_i)^2}{2\sigma_i^2} - \ln \sigma_i - \tfrac{1}{2}\ln 2\pi \Big].$$

The spread is learned as $\ln \sigma$ so it stays positive. The randomness is what lets the policy explore, and what makes $J(\theta)$ a smooth function of $\theta$ even when the dynamics are not. The planner's head computes the log-density and the entropy $H = \sum_i \big(\ln \sigma_i + \tfrac12 \ln 2\pi e\big)$; SciPy computes the same quantities independently.

In [ ]:
theta_head = {"log_std": jnp.array([-0.5])}  # sigma = exp(-0.5)
mu = jnp.array([0.7])
sigma = float(jnp.exp(theta_head["log_std"][0]))
u_values = jnp.array([[-1.0], [0.2], [0.7], [2.5]])

log_pi = jax.vmap(head.log_prob, in_axes=(None, None, 0))(theta_head, mu, u_values)
print("ln pi, planner:", np.round(np.asarray(log_pi), 6))
print("ln pi, SciPy:  ", np.round(norm.logpdf(np.asarray(u_values[:, 0]), loc=0.7, scale=sigma), 6))
print(f"entropy, planner: {float(head.entropy(theta_head)):.6f}   SciPy: {norm.entropy(scale=sigma):.6f}")

## 3. The policy gradient theorem, by sampling

Seeing the expected cost as a function of the parameters, gradient descent reads $\theta \leftarrow \theta - \eta \nabla_\theta J(\theta)$, and the policy gradient theorem gives that gradient without the model:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\big[\nabla_\theta \ln \pi_\theta(u \mid x)\; Q^{\pi_\theta}(x, u)\big].$$

It needs only samples of the cost and the gradient of the log-density: never $\nabla c$, never $f$.

The theorem can be checked where the exact gradient is known. Take one decision with no state, $u \sim \mathcal{N}(\mu, \sigma^2)$ and cost $c(u) = (u - u^\star)^2$, so that $Q = c$ and

$$J(\mu, \sigma) = (\mu - u^\star)^2 + \sigma^2, \qquad \frac{\partial J}{\partial \mu} = 2(\mu - u^\star), \qquad \frac{\partial J}{\partial \ln \sigma} = 2\sigma^2.$$

Below, each sample contributes $c(u)\, \nabla_\theta \ln \pi_\theta(u)$, with the draw $u$ held fixed while differentiating; the mean over many samples matches the exact gradient.

In [ ]:
U_STAR = 1.5
theta = {"mu": jnp.array([0.2]), "head": {"log_std": jnp.array([-0.5])}}
N = 20_000
keys = jax.random.split(jax.random.PRNGKey(1), N)


def stage_cost(u):
    return (u[0] - U_STAR) ** 2


def exact_J(theta):
    return (theta["mu"][0] - U_STAR) ** 2 + jnp.exp(2.0 * theta["head"]["log_std"][0])


def score_sample(theta, key, baseline=0.0):
    # One draw u ~ pi, then (c(u) - b) times the gradient of ln pi at that fixed draw
    u = jax.lax.stop_gradient(head.sample(theta["head"], theta["mu"], key))
    grad_log_pi = jax.grad(lambda th: head.log_prob(th["head"], th["mu"], u))(theta)
    return jax.tree_util.tree_map(lambda g: (stage_cost(u) - baseline) * g, grad_log_pi)


exact = jax.grad(exact_J)(theta)
samples = jax.vmap(score_sample, in_axes=(None, 0))(theta, keys)
for name, path in (("dJ/dmu", ("mu",)), ("dJ/dln sigma", ("head", "log_std"))):
    e, s = exact, samples
    for p in path:
        e, s = e[p], s[p]
    print(f"{name:13s} exact {float(e[0]):8.4f}   sampled {float(s.mean()):8.4f} +- {float(s.std() / np.sqrt(N)):.4f}")

## 4. REINFORCE

REINFORCE is the Monte Carlo version of the theorem: along an episode, the observed discounted cost-to-go $q_k^{MC} = \sum_{t \ge k} \alpha^{t-k} c_t$ stands in for $Q^\pi(x_k, u_k)$, and each step moves the parameters by

$$\theta \leftarrow \theta - \eta\; q_k^{MC}\, \nabla_\theta \ln \pi_\theta(u_k \mid x_k).$$

Each update uses a single sample of the gradient. On the one-decision problem that is one of the samples above, and its spread is the weakness of the method: the estimator is unbiased, but a single sample points almost anywhere. Worse, the cost is always positive, so the sign of one sample is only the sign of its exploration noise: every draw pushes the policy *away* from the action it tried, and only the weighting by the cost makes the average point the right way.

In [ ]:
single = np.asarray(samples["mu"][:, 0])
exact_mu = float(exact["mu"][0])
print(f"exact dJ/dmu = {exact_mu:.3f}; one REINFORCE sample: std {single.std():.3f}, "
      f"wrong sign in {100 * np.mean(np.sign(single) != np.sign(exact_mu)):.0f}% of draws")

plt.figure(figsize=(7, 3))
plt.hist(single, bins=120, range=(-40, 20), density=True, alpha=0.7, label="single-sample estimates")
plt.axvline(exact_mu, color="k", lw=2, label="exact gradient")
plt.xlabel("estimate of dJ/dmu")
plt.legend()
plt.show()

## 5. The baseline and the advantage

Subtracting from the cost any quantity $b(x)$ that does not depend on the action leaves the gradient unchanged in expectation, because the expected score is zero:

$$\mathbb{E}_{\pi_\theta}\big[\nabla_\theta \ln \pi_\theta(u \mid x)\big] = \int \nabla_\theta \pi_\theta(u \mid x)\, du = \nabla_\theta \int \pi_\theta(u \mid x)\, du = \nabla_\theta 1 = 0.$$

It does change the variance. The usual choice is the cost-to-go of the policy itself, $b(x) = J^\pi(x)$, which turns $Q$ into the **advantage**

$$A^\pi(x, u) = Q^\pi(x, u) - J^\pi(x),$$

how much better or worse this action is than the policy's average; in cost form a negative advantage is a good action. On the one-decision problem $J^\pi = J(\mu, \sigma)$: the score averages to zero, and subtracting it leaves the mean where it was while shrinking the spread.

In [ ]:
grad_log_pi = jax.vmap(
    lambda key: jax.grad(
        lambda th: head.log_prob(
            th["head"], th["mu"], jax.lax.stop_gradient(head.sample(theta["head"], theta["mu"], key))
        )
    )(theta)
)(keys)
print(f"mean score E[d ln pi / dmu] = {float(grad_log_pi['mu'].mean()):.4f} +- {float(grad_log_pi['mu'].std() / np.sqrt(N)):.4f}")

b = float(exact_J(theta))
with_baseline = jax.vmap(score_sample, in_axes=(None, 0, None))(theta, keys, b)
advantage_mu = np.asarray(with_baseline["mu"][:, 0])
wrong = lambda s: 100 * np.mean(np.sign(s) != np.sign(exact_mu))
print(f"without baseline: mean {single.mean():7.3f}, std {single.std():6.3f}, wrong sign in {wrong(single):.0f}% of draws")
print(f"with b = J:       mean {advantage_mu.mean():7.3f}, std {advantage_mu.std():6.3f}, wrong sign in {wrong(advantage_mu):.0f}% of draws")

plt.figure(figsize=(7, 3))
plt.hist(single, bins=120, range=(-40, 20), density=True, alpha=0.5, label="cost")
plt.hist(advantage_mu, bins=120, range=(-40, 20), density=True, alpha=0.5, label="advantage, b = J")
plt.axvline(exact_mu, color="k", lw=2, label="exact gradient")
plt.xlabel("estimate of dJ/dmu")
plt.legend()
plt.show()

## 6. Actor-critic and generalized advantage estimation

If the best baseline is the policy's cost-to-go, it can be learned. Two approximators then work together: an **actor** that carries $\pi_\theta$, and a **critic** that estimates $J^\pi$. The critic's temporal-difference error

$$\delta_k = c_k + \alpha\, J(x_{k+1}) - J(x_k)$$

is a one-step estimate of the advantage, so the actor can update at every step instead of waiting for the end of an episode: REINFORCE becomes TD where it was Monte Carlo. Methods that run many actors in parallel on this pattern are known as A2C and A3C.

One step is low-variance but biased by the critic's errors; the full Monte Carlo sum is unbiased but noisy. **Generalized advantage estimation** interpolates with a parameter $\lambda$:

$$A_k^{\text{GAE}} = \sum_{l \ge 0} (\alpha\lambda)^l\, \delta_{k+l}, \qquad \lambda = 0:\ A_k = \delta_k, \qquad \lambda = 1:\ A_k = q_k^{MC} - J(x_k),$$

computed backwards in time as $A_k = \delta_k + \alpha\lambda A_{k+1}$, and reset where an episode ends. The planner works in reward form, where $V = -J$ and every $\delta$ and $A$ changes sign. Below, `gae` runs on a small random batch of two plants whose first episode ends at step 3, and is checked against both limits written as explicit loops.

In [ ]:
n_steps, n_envs = 6, 2
reward = jax.random.normal(jax.random.PRNGKey(3), (n_steps, n_envs))
value = jax.random.normal(jax.random.PRNGKey(4), (n_steps, n_envs))
done = jnp.zeros((n_steps, n_envs)).at[3, 0].set(1.0)  # plant 0's episode ends at step 3
last_value = jax.random.normal(jax.random.PRNGKey(5), (n_envs,))  # V(x_N), bootstraps the last step
batch = {"reward": reward, "value": value, "done": done}
alpha = 0.97

# lambda = 0: the one-step TD error
value_next = jnp.concatenate([value[1:], last_value[None]])
delta = reward + alpha * value_next * (1.0 - done) - value
advantage_0, _ = gae(batch, last_value, alpha, 0.0)
print("lambda = 0 equals the TD error:          ", bool(jnp.allclose(advantage_0, delta)))

# lambda = 1: the discounted return to the end of the batch, minus the baseline
G, returns = last_value, []
for k in reversed(range(n_steps)):
    G = reward[k] + alpha * G * (1.0 - done[k])
    returns.append(G)
returns = jnp.stack(returns[::-1])
advantage_1, _ = gae(batch, last_value, alpha, 1.0)
print("lambda = 1 equals the return minus V(x): ", bool(jnp.allclose(advantage_1, returns - value)))
print("PPO uses lambda =", ppo.gae_lambda)

## 7. PPO: a trust region by clipping

Nothing in a plain gradient step stops an update from moving the policy too far. A step that is too large can destroy a performance learned slowly, and since the next data are generated by the new policy, the agent may never recover. The idea of a **trust region** is to limit how much the policy may change per update. PPO obtains the effect by clipping the probability ratio

$$\rho = \frac{\pi_\theta(u \mid x)}{\pi_{\theta_\text{old}}(u \mid x)}$$

between the policy being optimized and the one that collected the data. In reward form it maximizes

$$L^{\text{CLIP}}(\theta) = \mathbb{E}\Big[\min\big(\rho A,\ \text{clip}(\rho, 1 - \epsilon, 1 + \epsilon)\, A\big)\Big],$$

which, with the cost advantage $A_c = -A$, is minimizing $\mathbb{E}\big[\max(\rho A_c,\ \text{clip}(\rho) A_c)\big]$. Once the ratio has moved by more than $\epsilon$ in the direction that improves the objective, the gradient vanishes, while a move in the harmful direction is never clipped. That one line is why PPO is robust to its settings, simple to implement, and fit for continuous actions.

In [ ]:
eps = ppo.clip_range
ratio = np.linspace(0.0, 2.0, 400)
clipped = np.clip(ratio, 1 - eps, 1 + eps)
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, A, title in ((axes[0], 1.0, "A > 0: a better action"), (axes[1], -1.0, "A < 0: a worse action")):
    ax.plot(ratio, ratio * A, "--", color="gray", label="rho A, unclipped")
    ax.plot(ratio, np.minimum(ratio * A, clipped * A), lw=2, label="min(rho A, clip(rho) A)")
    ax.axvspan(1 - eps, 1 + eps, color="green", alpha=0.1, label="trust region")
    ax.set_xlabel("rho")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
axes[0].legend()
plt.show()

The complete loss the planner minimizes on each minibatch adds a regression of the critic onto the observed returns and an entropy bonus that keeps exploring:

$$\mathcal{L}(\theta) = -L^{\text{CLIP}}(\theta) + c_v\, \mathbb{E}\big[(R - V_\theta(x))^2\big] - c_H\, H(\pi_\theta),$$

with the advantages normalized over the minibatch. Below, `ppo.loss` is evaluated on a hand-made minibatch drawn from an older policy, and the same quantity is computed by hand from the head, the critic and the formula.

In [ ]:
params = ppo.params(planner.train_state)
functions = ppo.functions
n = 512
x = jax.random.uniform(jax.random.PRNGKey(6), (n, 2), minval=jnp.array([-np.pi, -1.0]), maxval=jnp.array([np.pi, 1.0]))
mu = jax.vmap(functions.mean, in_axes=(None, 0))(params["actor"], x)
a = jax.vmap(head.sample, in_axes=(None, 0, 0))(params["head"], mu, jax.random.split(jax.random.PRNGKey(7), n))
log_pi = jax.vmap(head.log_prob, in_axes=(None, 0, 0))(params["head"], mu, a)
minibatch = {
    "x": x,
    "a": a,
    "logp": log_pi + 0.3 * jax.random.normal(jax.random.PRNGKey(8), (n,)),  # an older policy
    "advantage": jax.random.normal(jax.random.PRNGKey(9), (n,)),
    "return": jax.random.normal(jax.random.PRNGKey(10), (n,)),
}
loss, stats = ppo.loss(params, minibatch)

# The same loss by hand
A = minibatch["advantage"]
A = (A - A.mean()) / (A.std() + 1e-8)
rho = jnp.exp(log_pi - minibatch["logp"])
L_clip = jnp.mean(jnp.minimum(rho * A, jnp.clip(rho, 1 - eps, 1 + eps) * A))
V = jax.vmap(functions.value, in_axes=(None, 0))(params["critic"], x)
by_hand = -L_clip + ppo.vf_coef * jnp.mean((minibatch["return"] - V) ** 2) - ppo.ent_coef * head.entropy(params["head"])

print(f"ppo.loss: {float(loss):.10f}")
print(f"by hand:  {float(by_hand):.10f}")
print(f"samples outside the trust region: {100 * float(jnp.mean(jnp.abs(rho - 1) > eps)):.0f}%")

## 8. The planner puts the pieces together

`ReinforcementLearningPlanner` repeats three steps. It **collects** a batch from many plants in parallel: states, actions drawn from the Gaussian head, their log-densities, the critic's values and the rewards. It **estimates** the advantages and returns with `gae`. It **updates** actor and critic by several epochs of minibatch gradient steps on `ppo.loss`. Each is the piece checked above; nothing else is needed to learn the swing-up.

In [ ]:
plan = planner.solve(timesteps=120_000)
print(plan.metadata.message, f"in {plan.metadata.solve_time_s:.1f} s")
planner.plot_learning_curve()
plt.show()
planner.get_controller().plot_control_law(x_axis=0, y_axis=1, u_axis=0)

## Summary

- A stochastic policy makes the expected cost a smooth function of its parameters, and the policy gradient theorem gives its gradient from samples alone, without the model.
- REINFORCE uses that gradient with Monte Carlo returns: unbiased, but so noisy that a single sample often points the wrong way.
- A baseline that does not depend on the action keeps the gradient and cuts its variance; the policy's own cost-to-go turns the cost into the advantage.
- Actor-critic learns that baseline, and generalized advantage estimation trades the critic's bias against Monte Carlo variance with $\lambda$.
- PPO clips the probability ratio so an update stays in a trust region around the policy that collected the data.

## Cheat sheet

| Concept | Formula, cost form | In the planner |
| --- | --- | --- |
| Stochastic policy | $u \sim \mathcal{N}(\mu_\theta(x), \sigma^2)$ | `head.sample`, `head.log_prob`, `head.entropy` |
| Objective | $J(\theta) = \mathbb{E}\big[\sum_k \alpha^k c_k\big]$ | reward $r = -c$ from `env.step`, discount `planner.gamma` |
| Policy gradient | $\nabla_\theta J = \mathbb{E}\big[\nabla_\theta \ln \pi_\theta\, Q^\pi\big]$ | `jax.grad` of the log-density |
| Baseline | $\mathbb{E}\big[\nabla_\theta \ln \pi_\theta\big] = 0$ | the critic's value |
| Advantage | $A^\pi = Q^\pi - J^\pi$ | the `advantage` of a batch, reward form |
| TD error | $\delta_k = c_k + \alpha J(x_{k+1}) - J(x_k)$ | `gae` with $\lambda = 0$ |
| GAE | $A_k = \delta_k + \alpha\lambda A_{k+1}$ | `gae`, $\lambda$ = `ppo.gae_lambda` |
| Clipped objective | $\max\big(\rho A_c,\ \text{clip}(\rho, 1 \pm \epsilon) A_c\big)$ | `ppo.loss`, $\epsilon$ = `ppo.clip_range` |
| Training loop | collect, estimate, update | `ReinforcementLearningPlanner.solve` |